# Customer Value Analysis — Data Cleaning & Feature Engineering

This notebook prepares the customer shopping dataset for downstream SQL analysis and Power BI reporting.

## Workflow
1. Load the source CSV.
2. Inspect structure, missing values, duplicate rows, and descriptive statistics.
3. Remove duplicate rows.
4. Create `Customer_Value_Score` and `Geography_Strength_Index`.
5. Export the processed dataset.

**Input expected:** `data/final_dataset.csv`  
**Output:** `outputs/final_dataset_cleaned.csv`

> Keep raw customer-level data private unless you have permission to publish it. This notebook does not include the dataset itself.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_PATH = Path("data") / "final_dataset.csv"
OUTPUT_PATH = Path("outputs") / "final_dataset_cleaned.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_PATH}. "
        "Place your source CSV at data/final_dataset.csv before running this notebook."
    )

df = pd.read_csv(INPUT_PATH)

print("Dataset preview:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values by column:")
display(df.isna().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nNumeric summary:")
display(df.describe())

print("\nCategorical summary:")
display(df.describe(include="object"))


In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Create customer-level value proxy used in this project
df["Customer_Value_Score"] = (
    df["Purchase Amount (USD)"] * df["Previous Purchases"]
)

# Calculate mean customer value score by location
df["Geography_Strength_Index"] = (
    df.groupby("Location")["Customer_Value_Score"].transform("mean")
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"Processed dataset saved to: {OUTPUT_PATH}")
print("\nFinal dataset preview:")
display(df.head())

print("\nFinal columns:")
print(df.columns.tolist())


## Notes

- `Customer_Value_Score` is a constructed proxy, defined here as purchase amount multiplied by previous purchases; it is not a verified lifetime-value measure.
- `Geography_Strength_Index` is the mean `Customer_Value_Score` for customers in the same location.
- Missing values are inspected but are not automatically imputed in this notebook.
- Place the input CSV in the expected path before running the notebook.
